In [2]:
from tqdm import tqdm
import numpy as np

import mpmath as mp
import sympy as sp
from sympy import *

import scipy.sparse as spmat
import scipy.sparse.linalg as spla

import matplotlib.pyplot as plt

In [3]:
x = symbols('x', real=True)
lam = symbols('lambda', real=True, positive=True)

In [4]:
def get_wavefunction_hyper(n):
    a = -n
    b = n + 2*lam
    c = lam + Rational(1, 2)
    z = (1 + sin(x)) / 2

    psi = (cos(x)) ** lam * hyper([a, b], [c], z)
    norm = sqrt( (2*(n + lam) * gamma(n + 2*lam))
                 / (factorial(n) * 2**(2*lam) * gamma(lam + Rational(1, 2))**2) )
    return (norm * psi)

def get_wavefunction_jacobi(n):
    norm = sqrt(2 * (n+lam) * gamma(n + 2 * lam) * gamma(n + 1) )/ (2 ** lam * gamma(n + lam + Rational(1, 2)))
    alpha = lam - Rational(1,2)
    psi = (cos(x))**lam * jacobi(n, alpha, alpha, -sin(x))

    return (norm * psi)

## Trigonometric Case

In [ ]:
def raise_trigonometric(n, psi):
    if n < 0:
        return psi
    result = -I * diff(psi, x) + I * (lam + n) * tan(x) * psi
    return raise_trigonometric(n - 1, result)

def get_aux_ground_state_trigonometric(n):
    norm = sqrt(gamma(lam + n + 1) / (sqrt(pi) * gamma(lam + n + Rational(1, 2))))
    return norm * cos(x)**(lam + n)

def get_wavefunction_raising(n):
    psi_0 = get_aux_ground_state_trigonometric(n)
    norm = sqrt(gamma(2 * lam + n) / (gamma(n+1) * gamma(2 * lam + 2 * n)))
    return (norm * raise_trigonometric(n - 1, psi_0))

In [ ]:
mp.mp.dps = 15
mp.mp.prec = mp.mp.dps * 3.33


In [ ]:
import mpmath as mp

def V_trig(x, lam):
    return lam*(lam-1)/mp.cos(x)**2 - lam**2

# ODE system
def schrodinger_trig_rhs(x, Y, lam, En):
    psi, psi_p = Y
    dpsi = psi_p
    dpsi_p = (V_trig(x, lam) - En) * psi
    return [dpsi, dpsi_p]

def shooting_trig(n, lam, eps=1e-6):
    En = n*(2*lam + n)
    
    x0 = -mp.pi/2 + eps
    psi0 = mp.cos(x0)**lam
    psi0_p = lam * mp.cos(x0)**(lam-1) * mp.sin(x0)
    
    sol = mp.odefun(
        lambda x, Y: schrodinger_trig_rhs(x, Y, lam, En),
        x0,
        [psi0, psi0_p]
    )
    
    return lambda x: sol(x)[0]


In [ ]:
def make_fun(expr):
    return lambdify(x, simplify(expr), 'mpmath')

def normalize_shooting(psi_raw, x0, x1):
    norm = mp.sqrt(mp.quad(lambda t: abs(psi_raw(t))**2, [x0, x1]))
    return lambda x: psi_raw(x) / norm

def compare_methods(n, N=500, lam_val=3):
    psi_s_raw = shooting_trig(n, lam_val)
    psi_h = make_fun(get_wavefunction_hyper(n).subs(lam, lam_val))
    psi_j = make_fun(get_wavefunction_jacobi(n).subs(lam, lam_val))

    x0 = -mp.pi/2 + 1e-6
    x1 =  mp.pi/2 - 1e-6
    xs = [x0 + i*(x1 - x0)/(N - 1) for i in range(N)]
    psi_s = normalize_shooting(psi_s_raw, x0, x1)

    # compute max differences of magnitude squared
    err_rh = max(abs(abs(psi_s(x))**2 - abs(psi_h(x))**2) for x in xs)
    err_rj = max(abs(abs(psi_s(x))**2 - abs(psi_j(x))**2) for x in xs)
    err_hj = max(abs(abs(psi_h(x))**2 - abs(psi_j(x))**2) for x in xs)

    print(f"n={n}")
    print("max |shooting - hyper| =", err_rh)
    print("max |shooting - jacobi| =", err_rj)
    print("max |hyper - jacobi|  =", err_hj)
    return err_rh, err_rj, err_hj

In [ ]:
err_rh, err_rj, err_hj = [], [], []
for n in range(101):
    rh, rj, hj = compare_methods(n)
    err_rh.append(rh)
    err_rj.append(rj)
    err_hj.append(hj)

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 6))

n_values = range(len(err_rh))
plt.semilogy(n_values, err_rh, 'b-', linewidth=2, label='|shooting - hyper|')
plt.semilogy(n_values, err_rj, 'r-', linewidth=2, label='|shooting - jacobi|')
plt.semilogy(n_values, err_hj, 'g--', linewidth=2, label='|hyper - jacobi|')

# Add grid for readability on log scale
plt.grid(True, which='both', alpha=0.3, linestyle='--')

# Labels and title
plt.xlabel('Quantum Number $n$', fontsize=12)
plt.ylabel('Max $|\psi_n|^2$ Difference', fontsize=12)
plt.title('Numerical Error Comparison of Wavefunction Methods ($\\lambda = 3$)', fontsize=14)
plt.legend(fontsize=11, loc='best')

# Set reasonable y-axis limits to show the full range
plt.ylim(1e-16, 1e40)

# Adjust layout and save
plt.tight_layout()

In [ ]:
n = 100

# symbolic difference
diff_expr = (get_wavefunction_hyper(n) - get_wavefunction_jacobi(n)).subs(lam, 3)

# convert to numeric function
diff_fun = lambdify(x, diff_expr, 'mpmath')

xs = np.linspace(-np.pi/2 + 1e-6, np.pi/2 - 1e-6, 1000)

# evaluate difference
ys = [abs(diff_fun(i)) for i in xs]

# plot
plt.figure(figsize=(8,4))
plt.plot(xs, ys)
plt.xlabel('x')
plt.ylabel('|ψ_hyper - ψ_jacobi|')
plt.title('Deviation of Hypergeometric vs Jacobi Methods (n=100, λ=3)')
plt.grid(True)
plt.show()

In [ ]:
Integral(abs(get_wavefunction_hyper(100).subs(lam, 3))**2, (x, -pi/2, pi/2)).evalf()

## Time Evolution

In [6]:
def get_trig_eigenstate(n):
    En = n * (2 * lam + n)
    psi = get_wavefunction_jacobi(n)
    return (En, psi)

In [7]:
def change_basis_trigonometric(psi_0_expr, lam_val, max_n=10, precision=4):
    domain = np.linspace(-np.pi/2, np.pi/2, 10**precision)
    psi_0 = lambdify(x, psi_0_expr.subs(lam, lam_val), 'numpy')

    basis = []
    for n in tqdm(range(max_n + 1), desc="Spectral basis projection"):
        energy, psi_n_expr = get_trig_eigenstate(n)
        energy = float(energy.evalf(subs={lam: lam_val}))

        psi_n = lambdify(x, sp.conjugate(psi_n_expr.subs(lam, lam_val)), 'numpy')
        coeff = np.trapz(psi_n(domain) * psi_0(domain), x=domain)

        psi_n_forward = lambdify(x, psi_n_expr.subs(lam, lam_val), 'numpy')
        basis.append((energy, coeff, psi_n_forward))

    return basis, domain

def time_evolve_trigonometric(basis, domain, max_t, frames=200):
    def psi_t(t):
        acc = np.zeros_like(domain, dtype=complex)
        for energy, coeff, psi_n in basis:
            acc += coeff * psi_n(domain) * np.exp(-1j * energy * t)
        return acc
    dx = domain[1] - domain[0]
    results = []
    for t in tqdm(np.linspace(0, max_t, frames), desc="Spectral evolution"):
        psi_val = psi_t(t)
        x_expect = np.trapz(np.conjugate(psi_val)*domain*psi_val, x=domain).real
        p_expect = np.trapz(np.conjugate(psi_val) * -1j * np.gradient(psi_val, dx))
        results.append((t, psi_val, x_expect, p_expect))
    return results

In [8]:
def trotter_time_evolve(domain, psi0_vals, Vx, max_t, frames=1000):
    N = len(psi0_vals)
    dx = domain[1] - domain[0]

    # Fourier frequencies
    k = np.fft.fftfreq(N, d=dx) * 2*np.pi

    dt = max_t/frames
    psi = psi0_vals.copy()

    results = []
    for j in tqdm(range(frames+1), desc="Trotter evolution"):
        t = j*dt
        dpsi_dx = np.gradient(psi, dx)
        d2psi_dx2 = np.gradient(dpsi_dx, dx)
        x_expect = np.trapz(np.conjugate(psi)*domain*psi, x=domain).real
        p_expect = np.trapz(np.conjugate(psi) * -1j * dpsi_dx, x=domain).real
        T_expect = np.trapz(np.conjugate(psi) * (-d2psi_dx2), x=domain).real
        V_expect = np.trapz(np.conjugate(psi) * Vx * psi, x=domain).real
        E_expect = T_expect + V_expect
        results.append((t, psi.copy(), x_expect, p_expect, T_expect, V_expect, E_expect))

        # Strang splitting (2nd order)
        psi = np.exp(-1j * Vx * (dt/2)) * psi
        psi = np.fft.ifft( np.exp(-1j * (k**2) * dt) * np.fft.fft(psi) )
        psi = np.exp(-1j * Vx * (dt/2)) * psi

    return results

In [9]:
def crank_nicolson(domain, psi_0_vals, Vx, max_t, frames=1000):
    psi = psi_0_vals
    dx = domain[1] - domain[0]
    N = len(domain)

    main = 2.0*np.ones(N)/dx**2 + Vx
    off  = -1.0*np.ones(N-1)/dx**2

    H = spmat.diags([off, main, off], [-1,0,1], format='csc')

    dt = max_t/frames
    I = spmat.identity(N, format="csc")

    A = I + 1j*(dt/2)*H
    B = I - 1j*(dt/2)*H
    A_factor = spla.factorized(A)

    results = []

    for j in tqdm(range(frames+1), desc="CN"):
        t = j*dt

        dpsi_dx = np.gradient(psi, dx)
        d2psi_dx2 = np.gradient(dpsi_dx, dx)
        x_expect = np.trapz(np.conjugate(psi)*domain*psi, x=domain).real
        p_expect = np.trapz(np.conjugate(psi) * -1j * dpsi_dx, x=domain).real
        T_expect = np.trapz(np.conjugate(psi) * (-d2psi_dx2), x=domain).real
        V_expect = np.trapz(np.conjugate(psi) * Vx * psi, x=domain).real
        E_expect = T_expect + V_expect
        results.append((t, psi.copy(), x_expect, p_expect, T_expect, V_expect, E_expect))

        rhs = B.dot(psi)
        psi = A_factor(rhs)

    return results


In [10]:
x0 = 0. # position displacement
p0 = 1 # momentum displacement
lam_val = 3
max_t = 10
psi_0 = sqrt(gamma(lam_val+1)/(sqrt(pi)*gamma(lam_val+Rational(1, 2)))) * (cos(x - x0)) ** lam_val
psi_0 *= exp(I * p0 * x)

In [11]:
basis, domain = change_basis_trigonometric(psi_0, lam_val)

Spectral basis projection: 100%|██████████| 11/11 [00:00<00:00, 67.27it/s]


In [12]:
cs = np.array([basis[i][1] for i in range(len(basis))])

In [19]:
basis

[(0.0,
  (0.9312837812920048-1.214306433183765e-17j),
  <function _lambdifygenerated(x)>),
 (7.0,
  (-1.474514954580286e-17-0.3535533905932739j),
  <function _lambdifygenerated(x)>),
 (16.0,
  (-0.08745316112428878+3.469446951953614e-18j),
  <function _lambdifygenerated(x)>),
 (27.0,
  (3.469446951953614e-18+1.734723475976807e-18j),
  <function _lambdifygenerated(x)>),
 (40.0,
  (-0.007680707306044109+1.0408340855860843e-17j),
  <function _lambdifygenerated(x)>),
 (55.0,
  (3.2959746043559335e-17+6.938893903907228e-18j),
  <function _lambdifygenerated(x)>),
 (72.0,
  (-0.0017493003659428625-1.214306433183765e-17j),
  <function _lambdifygenerated(x)>),
 (91.0,
  (6.938893903907228e-18-1.734723475976807e-17j),
  <function _lambdifygenerated(x)>),
 (112.0,
  (-0.0005793502430726655-6.938893903907228e-18j),
  <function _lambdifygenerated(x)>),
 (135.0,
  (2.531612072753653e-17-2.0816681711721685e-17j),
  <function _lambdifygenerated(x)>),
 (160.0,
  (-0.00023768692315076717+5.2041704279304

In [15]:
np.abs(cs[0])

0.9312837812920048

In [43]:
np.sum(np.abs(cs) ** 2)

0.9999999821464346

In [10]:
# Periodic domain with infinite walls setup
domain = np.linspace(-np.pi/2, np.pi/2, int(1e3))
Vx = lam_val*(lam_val - 1)/np.cos(domain)**2
Vx = Vx.clip(min(Vx)-1, 10**(lam_val+1))
psi_0_vals = lambdify(x, psi_0, 'numpy')(domain+x0)
psi_0_vals = np.roll(psi_0_vals, int(x0/(domain[1] - domain[0])))

In [15]:
# Finite potential well setup
domain = np.linspace(-np.pi, np.pi, int(1e3))
Vx = lam_val*(lam_val - 1)/np.cos(domain)**2
Vx = Vx.clip(min(Vx)-1, 10**(lam_val-1))
Vx[domain < -np.pi/2] = Vx[domain < -np.pi/2][-1]
Vx[domain > np.pi/2] = Vx[domain > np.pi/2][0]
psi_0_vals = lambdify(x, psi_0, 'numpy')(domain)
psi_0_vals[domain < -np.pi/2 + x0] = 0
psi_0_vals[domain > np.pi/2 + x0] = 0
psi_0_vals[0] = 0
psi_0_vals[-1] = 0

In [ ]:
plt.figure(figsize=(9, 4.5))

plt.plot(
    domain,
    Vx,
    color="black",
    linewidth=2.2,
    label=r"$V(x) = \lambda(\lambda-1)\sec^2 x$"
)

psi_scale = 0.75 * np.max(Vx)
psi_plot = psi_0_vals * psi_scale

plt.plot(
    domain,
    psi_plot,
    color="#1f77b4",
    linewidth=2.0,
    alpha=0.85,
    label=r"Initial wavefunction $\psi_0(x)$ (rescaled)"
)
plt.axhline(0, color="gray", linewidth=0.8, alpha=0.5)
plt.xlabel(r"Position $x$", fontsize=12)
plt.ylabel("Amplitude / Energy", fontsize=12)
plt.axvline(-np.pi/2, color="gray", linestyle="--", alpha=0.4)
plt.axvline(np.pi/2, color="gray", linestyle="--", alpha=0.4)
plt.legend(frameon=False, fontsize=11)
plt.grid(True, alpha=0.25)
plt.xlim(domain[0], domain[-1])
plt.ylim(min(Vx.min(), psi_plot.min()) * 1.05, max(Vx.max(), psi_plot.max()) * 1.05)
plt.title(
    rf"Initial State in Trigonometric Pöschl–Teller Potential ($\lambda={lam_val}$, $x_0={x0}$)",
    fontsize=13
)

plt.tight_layout()
plt.show()

In [ ]:
dx = domain[1] - domain[0]
dt = dx ** 2
T = int(max_t / dt)
crank = crank_nicolson(domain, psi_0_vals, Vx, max_t, frames=T)
trotter = trotter_time_evolve(domain, psi_0_vals, Vx, max_t, frames=T)

In [ ]:
from scipy.integrate import solve_ivp

def hamilton_eqs(_, y):
    x, p = y
    dxdt = 2 * p
    dpdt = - 2 * lam_val**2 * np.tan(x) / np.cos(x)**2
    return [dxdt, dpdt]

t_eval = np.linspace(0, max_t, T)

sol = solve_ivp(
    hamilton_eqs,
    t_span=(0, max_t),
    y0=[x0, p0],
    t_eval=t_eval,
    method="DOP853",
    rtol=1e-10,
    atol=1e-12
)

x_classical = sol.y[0]
p_classical = sol.y[1]
classical_V = lam_val**2/np.cos(x_classical)**2
classical_energy = p_classical**2 + classical_V

In [ ]:
import matplotlib.pyplot as plt

times = [r[0] for r in crank]
crank_x = [r[2] for r in crank]
crank_p = [r[3] for r in crank]
crank_T = [r[4] for r in crank]
crank_V = [r[5] for r in crank]
crank_E = [r[6] for r in crank]

trotter_x = [r[2] for r in trotter]
trotter_p = [r[3] for r in trotter]
trotter_T = [r[4] for r in trotter]
trotter_V = [r[5] for r in trotter]
trotter_E = [r[6] for r in trotter]

fig, axs = plt.subplots(3, 1, figsize=(10, 16), sharex=True)

# 1. Position
axs[0].plot(times, crank_x, label="Crank-Nicolson", linewidth=2)
axs[0].plot(times, trotter_x, label="Trotter", linewidth=1.6, alpha=0.85, c='g')
axs[0].plot(sol.t, x_classical, '--', color='black', linewidth=1.8, label="Classical")
axs[0].set_ylabel("Position ⟨x⟩", fontsize=12)
axs[0].legend()
axs[0].grid(True, alpha=0.25)
axs[0].set_title(f"Displacement x0={x0}, p0={p0}", fontsize=13)

# 2. Momentum
axs[1].plot(times, crank_p, label="Crank-Nicolson", linewidth=2)
axs[1].plot(times, trotter_p, label="Trotter", linewidth=1.6, alpha=0.85, c='g')
axs[1].plot(sol.t, p_classical, '--', color='black', linewidth=1.8, label="Classical")
axs[1].set_ylabel("Momentum ⟨p⟩", fontsize=12)
axs[1].legend()
axs[1].grid(True, alpha=0.25)

# 3. Total Energy
axs[2].plot(times, crank_E, label="Quantum Total Energy (CN)", linewidth=2)
axs[2].plot(times, trotter_E, label="Quantum Total Energy (Trotter)", linewidth=1.6, alpha=0.85, c='g')
axs[2].plot(sol.t, classical_energy, '--', color='black', linewidth=1.8, label="Classical Energy")
axs[2].set_ylabel("Total Energy", fontsize=12)
axs[2].legend()
axs[2].grid(True, alpha=0.25)

# 4. Potential Energy
# axs[3].plot(times, crank_V, label="Quantum Potential ⟨V⟩ (CN)", linewidth=2)
# axs[3].plot(times, trotter_V, label="Quantum Potential ⟨V⟩ (Trotter)", linewidth=1.6, alpha=0.85, c='g')
# axs[3].plot(sol.t, classical_V, '--', color='black', linewidth=1.8, label="Classical V(x)")
# axs[3].set_ylabel("Potential Energy", fontsize=12)
# axs[3].set_xlabel("Time", fontsize=12)
# axs[3].legend()
# axs[3].grid(True, alpha=0.25)

plt.tight_layout()
plt.show()
